# 📓 Semana 2 · Dia 1 — SQL avançado: CTEs, Window Functions e PIVOT

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (ELT with Spark SQL) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Exercícios de SQL avançado resolvidos |

---


## 📖 Teoria — Por que SQL avançado antes de Spark?

Spark SQL é o motor de transformação número 1 no Databricks — a prova DEA 2026 é chamada oficialmente de *ELT with Spark SQL and Python*. Quem domina SQL analítico avançado domina 60% das transformações de um pipeline sem escrever uma linha de Python.


## 📖 Teoria — CTEs e subqueries

Uma **CTE (Common Table Expression)** nomeia uma subconsulta para reutilização e legibilidade:

```sql
WITH receita_pais AS (
  SELECT Country, SUM(Quantity*UnitPrice) AS receita
  FROM vendas GROUP BY Country)
SELECT * FROM receita_pais WHERE receita > 100000;
```

**Regra mental**: WHERE filtra linhas antes da agregação; HAVING filtra o resultado depois. CTE é lida de cima para baixo — cada bloco é uma etapa do pipeline.


## 📖 Teoria — Window Functions — o que fazem

Uma window function calcula um valor **para cada linha** usando uma 'janela' de linhas relacionadas, SEM agrupar o resultado:

```sql
SELECT Country, InvoiceDate,
       ROW_NUMBER()   OVER (PARTITION BY Country ORDER BY InvoiceDate DESC) AS rn,
       RANK()         OVER (PARTITION BY Country ORDER BY receita DESC) AS rank,
       LAG(receita)   OVER (PARTITION BY Country ORDER BY mes) AS receita_mes_anterior,
       SUM(receita)   OVER (PARTITION BY Country ORDER BY mes) AS acumulado
FROM ...;
```

- `ROW_NUMBER()`: numeração sem empates
- `RANK()`: numeração com empates (pula posições)
- `DENSE_RANK()`: empates sem pular posições
- `LAG(col)`: valor da linha anterior na janela (para variação mês a mês)
- `LEAD(col)`: valor da próxima linha
- `SUM/AVG ... OVER`: agregação móvel/acumulada (running total)

> 🎯 **Dica de prova**: window functions são o tópico #1 em SQL avançado na DEA. Decore a diferença entre ROW_NUMBER / RANK / DENSE_RANK e o papel de PARTITION BY (fatiar) vs ORDER BY (ordenar dentro da fatia).


### 💻 Na prática — Na prática

Rode as consultas sobre a tabela Bronze e observe os resultados linha a linha.


In [ ]:
%sql
-- ROW_NUMBER: top 3 produtos por país
WITH vendas_prod AS (
  SELECT Country, StockCode, SUM(Quantity) AS qtd
  FROM workspace.bronze.vendas_bronze GROUP BY Country, StockCode)
SELECT Country, StockCode, qtd,
       ROW_NUMBER() OVER (PARTITION BY Country ORDER BY qtd DESC) AS rn
FROM vendas_prod QUALIFY rn <= 3
ORDER BY Country, rn

In [ ]:
%sql
-- LAG: variação de receita mês a mês
WITH receita_mes AS (
  SELECT DATE_TRUNC("month", InvoiceDate) AS mes,
         SUM(Quantity*UnitPrice) AS receita
  FROM workspace.bronze.vendas_bronze GROUP BY mes)
SELECT mes, receita,
       LAG(receita) OVER (ORDER BY mes) AS receita_anterior,
       ROUND((receita - LAG(receita) OVER (ORDER BY mes)) / LAG(receita) OVER (ORDER BY mes) * 100, 2)
         AS variacao_pct
FROM receita_mes ORDER BY mes

In [ ]:
%sql
-- PIVOT: vendas por país x trimestre
SELECT * FROM (
  SELECT Country,
         QUARTER(InvoiceDate) AS trimestre,
         Quantity * UnitPrice AS valor
  FROM workspace.bronze.vendas_bronze)
PIVOT (SUM(valor) FOR trimestre IN (1, 2, 3, 4))
ORDER BY Country LIMIT 5

> 🎯 **Dica de prova**: `QUALIFY` filtra o resultado de uma window function (como HAVING para GROUP BY) — cai com frequência. No Databricks, QUALIFY é suportado nativamente.


## 🎯 Exercícios de fixação

**1.** Qual o ticket médio por país usando CTE e window (média móvel de 3 meses por país)?

**2.** Liste os 2 produtos mais vendidos de cada país (use ROW_NUMBER + QUALIFY).

**3.** Calcule a receita acumulada por país ao longo dos meses (running total).

**4.** Diferencie ROW_NUMBER, RANK e DENSE_RANK com um exemplo de empate.


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Média móvel 3 meses

```sql
WITH rec AS (
  SELECT Country, DATE_TRUNC('month',InvoiceDate) mes, SUM(Quantity*UnitPrice) receita
  FROM workspace.bronze.vendas_bronze GROUP BY Country, mes)
SELECT Country, mes, AVG(receita) OVER (PARTITION BY Country ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) media_3m
FROM rec;
```

**2.** Top 2 por país

```sql
WITH vp AS (SELECT Country, StockCode, SUM(Quantity) qtd FROM workspace.bronze.vendas_bronze GROUP BY Country, StockCode)
SELECT * FROM (SELECT Country, StockCode, qtd, ROW_NUMBER() OVER (PARTITION BY Country ORDER BY qtd DESC) rn FROM vp) WHERE rn <= 2;
```

**3.** Running total

`SUM(receita) OVER (PARTITION BY Country ORDER BY mes)` — sem ROWS BETWEEN, o padrão é até a linha atual (cumulativo).

**4.** ROW_NUMBER vs RANK

Com empate (qtd 10,10,9): ROW_NUMBER dá 1,2,3; RANK dá 1,1,3; DENSE_RANK dá 1,1,2. RANK pula posição após empate; DENSE_RANK não.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*